# Health, Graceful Shutdown, Multi-Tenancy, and Audit

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 28 §28.6–28.8 · type: concept-lab*

A service doesn't just compute — it must be **operable** by the platform running it.
This lab makes the small, non-negotiable operability surface *tangible*: liveness vs.
readiness probes, a graceful drain that doesn't drop in-flight work, a **centralized**
tenant filter, and an append-only audit log. You'll *feel* why these cross-cutting rules
belong in **one** place — and why CQRS/event sourcing usually don't belong at all.

## 🧠 Why this matters

Two probes answer two different questions the orchestrator asks:

- **Liveness** — "am I alive?" Cheap, no dependencies. If it fails, restart the pod.
- **Readiness** — "am I ready for *traffic*?" Checks that dependencies are connected and
  caches are warm. If it fails, stop routing requests here (but don't restart).

Confuse them and you get cascading restarts (a slow DB fails liveness → the platform
kills healthy pods) or dropped traffic (readiness always "ok" → requests hit a pod whose
DB is down).

Then there's **graceful shutdown**: on a termination signal, *stop accepting new work,
finish in-flight work, flush and close* — so a deploy doesn't drop a user's request or
corrupt an agent run mid-step. And the **enterprise must-haves** — multi-tenancy, audit,
soft deletes — share one lesson: cross-cutting rules must be enforced in **one
hard-to-bypass place**, because the failure mode is 49 of 50 endpoints scoping by tenant
and the 50th leaking everyone's data.

## Objectives & prerequisites

**By the end you can:**

- Implement `/healthz` (liveness, cheap) and `/readyz` (readiness, checks a dependency)
  and explain when each should fail.
- Simulate **graceful shutdown**: drain in-flight tasks on a signal and show a request
  *survives* vs. a hard kill that drops it.
- Enforce **multi-tenancy** through a single scoped-query helper, and see how one missing
  filter leaks across tenants — then make the scoped path the default.
- Keep an **append-only audit log** (who/what/when) distinct from debug logs, and apply a
  **soft delete** (`deleted_at`) with the must-exclude-from-queries gotcha.

**Prereqs:** 28-02 (config, DI, flags). Ch 14 (checkpointing) is *referenced* for the
shutdown predict — no code from it is required here.

**Run first:** the Setup cell. Fully **offline**: "in-flight work" is `asyncio.sleep`
tasks and the DB is a mock, so the notebook is free and deterministic in any mode.

In [ ]:
# --- Setup -------------------------------------------------------------------
import asyncio
import os
import random
from dataclasses import dataclass, field
from datetime import datetime, timezone

from dotenv import load_dotenv

load_dotenv()

# Offline lab: probes hit a MOCK dependency, shutdown drains asyncio.sleep tasks,
# tenancy/audit run on in-memory rows. MOCK=1 (default) or 0 -> both free & offline.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(28)  # determinism for the simulated work below


def _now() -> str:
    # Fixed clock in MOCK so audit timestamps are deterministic for CI/readers.
    if MOCK:
        return "2026-01-01T00:00:00+00:00"
    return datetime.now(timezone.utc).isoformat()


print("Offline operability lab. No API key needed; nothing is billed.")

## 1 · Liveness vs. readiness: two probes, two questions

`/healthz` is **liveness** — it returns `ok` as long as the process is running, with no
dependency checks, so a transient DB hiccup can't trigger a needless restart. `/readyz`
is **readiness** — it actually pokes a dependency (here a mock `SELECT 1`) and reports
whether this instance should receive traffic.

We model the "DB" as a tiny object with a toggle so we can flip it down and watch each
probe respond differently.

In [ ]:
class MockDB:
    """Stands in for a real connection pool. `up=False` simulates an outage."""

    def __init__(self, up: bool = True):
        self.up = up

    async def execute(self, sql: str) -> int:
        await asyncio.sleep(0)  # yield, as a real driver would
        if not self.up:
            raise ConnectionError("database unavailable")
        return 1  # e.g. the result of SELECT 1


db = MockDB(up=True)


async def healthz() -> dict:
    # Liveness: cheap, NO dependency checks. Alive == process is running.
    return {"status": "ok"}


async def readyz() -> tuple[int, dict]:
    # Readiness: check the things we need to actually serve a request.
    try:
        await db.execute("SELECT 1")
        return 200, {"status": "ready"}
    except Exception as exc:  # noqa: BLE001 -- report, don't crash the probe
        return 503, {"status": "not ready", "reason": str(exc)}


print("healthz:", asyncio.run(healthz()))
print("readyz :", asyncio.run(readyz()))

## 2 · 🔮 Predict: the database goes down

Flip the mock DB to `up=False` and call **both** probes again.

**Predict before running:**
1. What does `/healthz` return now — `ok`, or an error?
2. What status code does `/readyz` return?
3. Given those answers, what should the orchestrator do: **restart** this pod, or just
   **stop routing traffic** to it? Why is that distinction the whole point?

In [ ]:
db.up = False  # simulate a dependency outage

print("healthz:", asyncio.run(healthz()), "  <- still OK: the PROCESS is fine")
status, body = asyncio.run(readyz())
print(f"readyz : {status} {body}  <- NOT ready: pull it from the load balancer")

print(
    "\nCorrect platform response: DO NOT restart (liveness is green) -- just stop\n"
    "routing traffic until readiness recovers. Failing liveness here would kill a\n"
    "healthy process and can cascade across the fleet."
)

db.up = True  # restore for the rest of the notebook

## 3 · Graceful shutdown: drain in-flight work instead of dropping it

On a termination signal (SIGTERM from the orchestrator during a deploy), a good service
**stops accepting new work**, lets in-flight work **finish**, then flushes and closes.
We model a server with a set of running tasks and a `draining` flag. Compare two
shutdowns of the *same* in-flight request: a **graceful drain** (awaits the task) vs. a
**hard kill** (cancels it).

In [ ]:
@dataclass
class Server:
    draining: bool = False
    _tasks: set = field(default_factory=set)
    completed: list = field(default_factory=list)
    rejected: list = field(default_factory=list)

    async def _handle(self, name: str, work_seconds: float):
        await asyncio.sleep(work_seconds)   # the in-flight work (an agent step, a query)
        self.completed.append(name)

    def accept(self, name: str, work_seconds: float = 0.05):
        if self.draining:                   # after a signal, refuse NEW work cleanly
            self.rejected.append(name)
            return None
        task = asyncio.create_task(self._handle(name, work_seconds))
        self._tasks.add(task)
        task.add_done_callback(self._tasks.discard)
        return task

    async def graceful_shutdown(self):
        self.draining = True                # 1) stop accepting new work
        if self._tasks:                     # 2) let in-flight work finish
            await asyncio.gather(*self._tasks, return_exceptions=True)
        # 3) (flush/close connections here in a real service)

    async def hard_kill(self):
        self.draining = True
        for t in list(self._tasks):         # cancel mid-flight -> work is DROPPED
            t.cancel()
        await asyncio.gather(*self._tasks, return_exceptions=True)


async def demo_graceful():
    s = Server()
    s.accept("request-A")                   # in-flight when the signal arrives
    await asyncio.sleep(0.01)               # signal lands mid-request
    await s.graceful_shutdown()             # drain
    s.accept("request-B")                   # arrives AFTER signal -> cleanly rejected
    return s


g = asyncio.run(demo_graceful())
print("graceful  -> completed:", g.completed, "| rejected-new:", g.rejected)

## 4 · The contrast: a hard kill drops the request

Same in-flight request, but we **cancel** instead of draining. The work never lands in
`completed` — that's a dropped user request or a half-finished agent run. Watching the
two side by side is the point of the lab.

In [ ]:
async def demo_hard_kill():
    s = Server()
    s.accept("request-A")
    await asyncio.sleep(0.01)
    await s.hard_kill()                      # cancel in-flight -> dropped
    return s


h = asyncio.run(demo_hard_kill())
print("hard kill -> completed:", h.completed, "  <- request-A was DROPPED")
print("\nGraceful drain kept the request; the hard kill lost it. A deploy should",
      "\nnever be a hard kill of healthy in-flight work.")

## ⚠️ Pitfall (and a 🔮 predict): a worker killed mid-agent-run

For long-running agents, graceful shutdown matters even more than for plain web
requests: a pod killed mid-run can leave a task half-finished. **Per-step
checkpointing** (Chapter 14) is what lets a restarted worker *resume* instead of losing
the run.

**Predict:** two agents are killed at step 2 of 4 — one checkpoints each step, one
doesn't. After a restart, which resumes from step 3, and which has to start over (or is
simply lost)?

In [ ]:
async def run_agent(steps: int, checkpoint: list | None, kill_at: int | None):
    """Run `steps`; if `checkpoint` is provided, persist progress each step. If
    `kill_at` is set, raise CancelledError after that step (simulated SIGTERM)."""
    done = checkpoint[-1] if checkpoint else 0   # resume from last checkpoint
    for step in range(done + 1, steps + 1):
        await asyncio.sleep(0)                   # the step's work
        if checkpoint is not None:
            checkpoint.append(step)              # <-- per-step durability
        if kill_at is not None and step == kill_at:
            raise asyncio.CancelledError(f"killed at step {step}")
    return f"completed all {steps} steps"


# Agent WITH checkpointing: killed at step 2, then restarted -> resumes at step 3.
ckpt = []
try:
    asyncio.run(run_agent(4, checkpoint=ckpt, kill_at=2))
except asyncio.CancelledError as e:
    print("checkpointed agent:", e, "| saved progress:", ckpt)
result = asyncio.run(run_agent(4, checkpoint=ckpt, kill_at=None))  # restart
print("  after restart ->", result, "| progress:", ckpt)

# Agent WITHOUT checkpointing: killed at step 2 -> nothing saved, run is lost.
try:
    asyncio.run(run_agent(4, checkpoint=None, kill_at=2))
except asyncio.CancelledError as e:
    print("uncheckpointed agent:", e, "| saved progress: NONE -> must restart from 0")

## 5 · Multi-tenancy: scope every query in *one* place

When one deployment serves many customers, **every** query must be scoped to a tenant.
The cheapest model is a shared table with a `tenant_id` on every row — fine, *as long as*
the filter is enforced centrally. We show the danger first: a raw query with no scope
returns **everyone's** rows. Then we route all reads through one `scoped_query` helper so
the tenant filter is the **default that's hard to bypass**.

In [ ]:
ROWS = [
    {"id": 1, "tenant_id": "acme",   "text": "Acme refund policy"},
    {"id": 2, "tenant_id": "globex", "text": "Globex refund policy"},
    {"id": 3, "tenant_id": "acme",   "text": "Acme agent config"},
    {"id": 4, "tenant_id": "initech","text": "Initech secrets"},
]


def unsafe_query(text_contains: str) -> list[dict]:
    # The 50th endpoint someone forgot to scope. Leaks ACROSS tenants.
    return [r for r in ROWS if text_contains.lower() in r["text"].lower()]


def scoped_query(tenant_id: str, text_contains: str = "") -> list[dict]:
    # The ONE helper every read goes through. tenant_id is required, first, always.
    return [
        r for r in ROWS
        if r["tenant_id"] == tenant_id and text_contains.lower() in r["text"].lower()
    ]


leak = unsafe_query("refund")
print("UNSAFE 'refund' ->", [(r["tenant_id"], r["text"]) for r in leak],
      "  <- LEAK: two tenants!")

acme_only = scoped_query("acme", "refund")
print("SCOPED acme 'refund' ->", [(r["tenant_id"], r["text"]) for r in acme_only])
assert all(r["tenant_id"] == "acme" for r in scoped_query("acme")), "tenant leak!"
print("Centralized scope holds: acme sees only acme rows.")

## 6 · Audit log + soft delete

An **audit log** is an append-only record of *who did what, to what, and when* — distinct
from debug logs, often a compliance requirement, always invaluable in an incident. A
**soft delete** marks a row `deleted_at` instead of removing it (recoverable, preserves
history) — with the classic gotcha: **you must exclude soft-deleted rows from normal
queries**, and still honor genuine deletion requests under privacy law.

In [ ]:
AUDIT: list[dict] = []  # append-only; never updated or deleted in place


def audit(actor: str, action: str, target: str) -> None:
    AUDIT.append({"who": actor, "action": action, "target": target, "when": _now()})


# A soft delete: mark, don't remove.
def soft_delete(row_id: int, actor: str) -> None:
    for r in ROWS:
        if r["id"] == row_id:
            r["deleted_at"] = _now()
            audit(actor, "soft_delete", f"row:{row_id}")


def scoped_live_query(tenant_id: str) -> list[dict]:
    # The gotcha made the default: soft-deleted rows are EXCLUDED from normal reads.
    return [r for r in ROWS
            if r["tenant_id"] == tenant_id and r.get("deleted_at") is None]


soft_delete(3, actor="user:alice")  # Acme deletes its agent-config row
audit("user:alice", "read", "tenant:acme")

live = scoped_live_query("acme")
print("acme live rows (soft-deleted excluded):",
      [r["text"] for r in live])           # 'Acme agent config' is gone from reads
print("row 3 still on disk for recovery:",
      any(r["id"] == 3 and r.get("deleted_at") for r in ROWS))
print("\nAudit trail:")
for e in AUDIT:
    print("  ", e)

## ⚠️ Pitfall: CQRS / event sourcing are seductive and over-applied

Before reaching for **CQRS** (split write model from read model) or **event sourcing**
(store state as an immutable event log), price what you pay *forever*. The cell lists the
permanent moving parts. For almost every service — including the capstone — **plain CRUD
on Postgres with the soft-delete + audit patterns above is the right answer.** Adopt CQRS
/ ES only when a concrete need (audit-by-design, extreme read/write asymmetry, temporal
queries) clearly justifies the cost.

In [ ]:
CQRS_ES_COSTS = [
    "Projections: read models must be rebuilt from events and kept in sync.",
    "Eventual consistency: the read side lags the write side -- UIs must tolerate it.",
    "Event-schema evolution: old events live forever; every change needs upcasting.",
    "Operational surface: more services, more failure modes, harder local dev.",
    "Debuggability: 'current state' is a replay, not a row you can SELECT.",
]
print("What CQRS/event sourcing costs you FOREVER:")
for c in CQRS_ES_COSTS:
    print("  -", c)
print("\nDefault: CRUD on Postgres + soft deletes + an append-only audit table.")

## 🎯 Senior lens: centralize the rule, make it hard to bypass

The enterprise failure mode is not a clever attack — it's **the 50th endpoint**. Forty-
nine routes scope by tenant; one forgets, and it leaks every customer's data. The same
shape repeats for audit, authz, rate limits, and soft-delete exclusion: a rule applied
*per-endpoint* will eventually be missed somewhere.

So a senior pushes these cross-cutting concerns into **one place that's the default** —
middleware, a base query, a repository layer the routes can't go around — and makes the
*unscoped* path the awkward one to write. You saw it here: `scoped_query`/
`scoped_live_query` require `tenant_id` and exclude soft-deletes by construction, so the
safe path is the easy path. That, not heroics, is how multi-tenant systems stay correct
across hundreds of endpoints and years of edits.

## Recap

- **Liveness vs. readiness** answer different questions: `/healthz` is cheap and
  dependency-free (failing it restarts you); `/readyz` checks dependencies (failing it
  pulls you from traffic). Don't conflate them.
- **Graceful shutdown** stops new work, drains in-flight work, then flushes/closes — so a
  deploy doesn't drop requests; a hard kill *does*.
- For long agents, **per-step checkpointing** (Ch 14) turns a kill into a *resume*
  instead of a lost run.
- **Multi-tenancy, audit, soft delete** all want **one centralized, hard-to-bypass**
  enforcement point; the failure mode is the single endpoint that forgot.
- **CQRS / event sourcing** are usually over-applied — default to **CRUD on Postgres**
  with audit + soft deletes.

## Exercises

Use the empty cells below. (Solutions land in `solutions/` in Phase 2.)

1. **Readiness with multiple deps.** Extend `readyz` to check both the DB *and* a mock
   cache, returning `503` plus *which* dependency failed. 🔮 Predict the body when only
   the cache is down.
2. **Shutdown timeout.** Add a `drain_timeout` to `graceful_shutdown`: drain up to N
   seconds, then cancel whatever's left and audit it as `forced_drain`. Predict what
   happens to a task that needs longer than the timeout.
3. **Plug the leak for real.** Make `unsafe_query` impossible to call by routing all
   reads through a `Repository` class whose only public method requires `tenant_id`.
   Argue (markdown) why a *type-level* requirement beats a code-review convention.

In [ ]:
# Exercise 1 -- your code here


In [ ]:
# Exercise 2 -- your code here


In [ ]:
# Exercise 3 -- your code here


## Next

You've built the operability surface: probes, drain, tenancy, audit. That's the full
shape of the chapter — and the shape of the capstone's `app/`.

- ◀️ **Previous:** [`28-02-config-di-and-feature-flags.ipynb`](./28-02-config-di-and-feature-flags.ipynb).
- 🧩 **Template:** these probes and the tenant/audit helpers feed
  [`../../../templates/fastapi-agent-service/`](../../../templates/fastapi-agent-service/).
- 🔧 **Blueprint:**
  [`../../../blueprints/fastapi-agent-service/`](../../../blueprints/fastapi-agent-service/)
  wires `/healthz` + `/readyz` and graceful shutdown into a real FastAPI app.
- 🎓 **Capstone:** this is `capstone/app/health.py`, the shutdown handler, and the
  tenant-scoped repository / audit log; checkpoint `checkpoints/ch28-app-architecture`.
- 📖 **Book:** see §28.6 (CQRS/event sourcing trade-offs), §28.7 (multi-tenancy, audit,
  soft deletes), §28.8 (health checks, graceful shutdown, the operational contract).
  Graceful agent shutdown ties back to §14 (checkpointing).